In [ ]:
from __future__ import annotations

import  operator
from typing import TypedDict, List, Annotated

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import PydanticOutputParser

from dotenv import load_dotenv
load_dotenv()

True

In [254]:
class Task(BaseModel):
    id: int
    title: str
    brief: str = Field(description="What to cover")

In [255]:
class Plan(BaseModel):
    blog_title: str
    task: List[Task]

In [256]:
class State(TypedDict):
    topic: str
    plan: Plan
    ## reducer: results from workers gets concatenated automatically here
    sections: Annotated[List[str], operator.add]
    final: str


In [257]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    provider="auto",  # let Hugging Face choose the best provider for you
)

chat_model = ChatHuggingFace(llm=llm)


In [ ]:
# 1. Create a parser for your Plan class
parser = PydanticOutputParser(pydantic_object=Plan)

def orchestrator(state: State) -> dict:
    # 2. Get instructions on how the model should format the JSON
    format_instructions = parser.get_format_instructions()

    # 3. Use the chat_model (the ChatHuggingFace wrapper)
    # We add the format instructions to the prompt manually
    prompt = [
        SystemMessage(content=(
            "You are a blog planner. Create a plan with 5-7 sections. "
            "Output ONLY the JSON. Do not include any thinking or intro text."
            f"\n\n{format_instructions}"
        )),
        HumanMessage(content=f"Topic: {state['topic']}")
    ]

    # 4. Chain the model with the parser
    chain = chat_model | parser
    
    # This will return a validated 'Plan' object
    full_plan = chain.invoke(prompt)
    
    return {"plan": full_plan}

In [259]:
def fanout(state: State):
    return [
        Send("worker", {"task": task, "topic": state["topic"], "plan": state["plan"]})
        for task in state["plan"].task
    ]

In [260]:
def worker(payload: dict) -> dict:

    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]

    blog_title = plan.blog_title

    section_md = chat_model.invoke([
        SystemMessage(content="Write one clean markdown section for a blog."),
        HumanMessage(
            content=(
                f"Blog Title: {blog_title}\n\n"
                f"Section Title: {task.title}\n\n"
                f"Brief: {task.brief}\n\n"
                "Return only section content in markdown."
            )
        )
    ]).content

    return {"sections": [section_md]}

In [261]:
from pathlib import Path

def reducer(state: State) -> dict:

    title = state["plan"].blog_title
    body = "\n\n".join(state["sections"]).strip()

    final_md = f"# {title}\n\n{body}\n"

    # save to file
    filename = title.lower().replace(" ", "_") + ".md"
    filepath = Path(filename)
    filepath.write_text(final_md, encoding="utf-8")

    return {"final": final_md}

In [262]:
graph = StateGraph(State)
graph.add_node("orchestrator", orchestrator)
graph.add_node("worker", worker)
graph.add_node("reducer", reducer)

In [263]:
graph.add_edge(START, "orchestrator")
graph.add_conditional_edges("orchestrator", fanout, ["worker"])
graph.add_edge("worker", "reducer")
graph.add_edge("reducer", END)

app = graph.compile()

In [264]:
## app

In [265]:
output = app.invoke({
    "topic": "Write a blog on Neural Networks.", 
    "sections": []
    })


In [266]:
print(output)

{'topic': 'Write a blog on Neural Networks.', 'plan': Plan(blog_title='A Comprehensive Guide to Neural Networks', task=[Task(id=1, title='Introduction to Neural Networks', brief='Overview of the basics of neural networks, including their history, components, and types.'), Task(id=2, title='Neural Network Architecture', brief='Explanation of the different layers and components of a neural network, including input, hidden, and output layers.'), Task(id=3, title='Training and Optimization', brief='Discussion of the different algorithms and techniques used to train and optimize neural networks, including backpropagation and stochastic gradient descent.'), Task(id=4, title='Deep Learning Techniques', brief='Examination of the different techniques used in deep learning, including convolutional neural networks (CNNs), recurrent neural networks (RNNs), and long short-term memory (LSTM) networks.'), Task(id=5, title='Applications of Neural Networks', brief='Discussion of the different applicati